# CORS, CSRF, and Rate Limiting

This notebook covers:

1. What the same-origin policy actually enforces, and what CORS relaxes
2. Configure `CORSMiddleware` with a strict allowlist (not `*`)
3. When CSRF protection matters (cookie auth) and when it doesn't (Bearer tokens)
4. Add a process-local rate limiter with `slowapi`
5. Apply different limits per route and per user; bypass for `/health`
6. Why a real production limiter needs a shared backend (Redis)

**Scope**: FastAPI + `slowapi` + `TestClient`. The rate limiter here is in-process — fine for tutorials and single-worker apps, useless across replicas. Section 6 explains exactly why.

## 1. The Same-Origin Policy and CORS

Every request a browser makes has an **origin**: `scheme + host + port`. So `https://app.example.com` and `https://api.example.com` are *different* origins (different host), and `http://example.com:8080` is a different origin from `http://example.com` (different port).

The **same-origin policy** is a browser rule: scripts loaded from origin A may *issue* a request to origin B, but cannot *read the response* unless B opts in. The point is to keep `evil.com`'s JavaScript from reading your bank's logged-in account page through a hidden `fetch()`.

**Three pieces of this matter for API design:**

- **Simple requests** (GET/POST with safe content types, no custom headers) go through directly — the browser sends them. It just gates the *response* behind the CORS check.
- **Preflight requests** are an `OPTIONS` the browser sends *before* a non-simple request (any custom header like `Authorization`, methods like `PUT`/`DELETE`, or `Content-Type: application/json`). The server must respond with `Access-Control-Allow-Origin`, `Access-Control-Allow-Methods`, etc. — if it doesn't, the browser refuses to send the real request.
- **`curl` and `httpx` ignore all of this.** CORS is a *browser* enforcement. A backend test with TestClient won't reproduce a CORS bug — you have to test from a browser (or by inspecting the headers, which is what we'll do).

**CORS** (Cross-Origin Resource Sharing) is the server's way of telling browsers "these other origins are okay." `CORSMiddleware` writes the `Access-Control-*` response headers for you.

**`Access-Control-Allow-Origin: *` is almost always wrong.** It works only for credential-less requests. The moment you need to send cookies or `Authorization`, the spec forbids the wildcard, and the browser silently drops the response. Use an explicit allowlist.

## 2. `CORSMiddleware` Configuration

FastAPI's `CORSMiddleware` (re-exported from Starlette) handles both the preflight `OPTIONS` and the `Access-Control-Allow-Origin` header on responses. Configure four knobs:

- **`allow_origins`** — a list of exact origin strings (`"https://app.example.com"`). Or `allow_origin_regex` for wildcards ("`https://.*\.example\.com`"). **Never** `["*"]` in production.
- **`allow_credentials`** — set `True` if your frontend sends cookies or `Authorization`. (With this on, `*` is rejected by the browser.)
- **`allow_methods`** — typically `["GET", "POST", "PUT", "PATCH", "DELETE"]`. `OPTIONS` is added automatically.
- **`allow_headers`** — `["Authorization", "Content-Type"]` covers most APIs. Custom headers your frontend uses need to be listed here or the preflight fails.

Below: a strict configuration, plus a TestClient demonstration of the preflight handshake. (Reminder: TestClient *simulates* the browser exchange; in production the *browser* enforces it.)

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://app.example.com", "http://localhost:5173"],  # explicit
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "PATCH", "DELETE"],
    allow_headers=["Authorization", "Content-Type"],
    max_age=600,  # cache the preflight result for 10 minutes
)

@app.get("/portfolios/me")
def me():
    return {"user": "alice"}

client = TestClient(app)

# 1) Preflight from an ALLOWED origin: the middleware mirrors back Access-Control-Allow-Origin.
preflight_ok = client.options(
    "/portfolios/me",
    headers={
        "Origin": "https://app.example.com",
        "Access-Control-Request-Method": "GET",
        "Access-Control-Request-Headers": "authorization",
    },
)
print("allowed origin preflight  :", preflight_ok.status_code)
print("  Allow-Origin            :", preflight_ok.headers.get("access-control-allow-origin"))
print("  Allow-Credentials       :", preflight_ok.headers.get("access-control-allow-credentials"))
print("  Allow-Headers           :", preflight_ok.headers.get("access-control-allow-headers"))

# 2) Preflight from a DISALLOWED origin: middleware refuses, no Allow-Origin header.
preflight_bad = client.options(
    "/portfolios/me",
    headers={
        "Origin": "https://evil.example.com",
        "Access-Control-Request-Method": "GET",
    },
)
print("\ndisallowed origin preflight:", preflight_bad.status_code)
print("  Allow-Origin             :", preflight_bad.headers.get("access-control-allow-origin"))
# Note: the *response* may still be 200/400, but without Allow-Origin the browser will refuse it.

# 3) Actual GET from the allowed origin: response carries Allow-Origin for the browser to validate.
r = client.get("/portfolios/me", headers={"Origin": "https://app.example.com"})
print("\nactual GET allowed         :", r.status_code, "-> Allow-Origin:", r.headers.get("access-control-allow-origin"))

Production-grade CORS configurations to internalize:

- **Never wildcard origins on a credentialed API.** The browser rejects `Allow-Origin: *` whenever `Allow-Credentials: true`. You'd silently lose CORS to your real users.
- **`max_age`** controls how long the browser caches the preflight (no second `OPTIONS` for the same method/headers combo for 10 minutes in our case). Worth setting — saves a round-trip per route per user per 10 minutes.
- **The middleware order matters.** CORS should run *before* auth: a CORS-rejected request shouldn't waste a JWT decode.
- **Test the preflight, not just the GET.** Most CORS bugs are an `OPTIONS` returning 400 because `allow_headers` is missing a custom header your frontend sends. A working `GET /docs` is no signal.

## 3. CSRF: When It's Relevant

CSRF (Cross-Site Request Forgery) is the inverse of the CORS threat. Picture this:

1. You're logged into `bank.com`. Your session cookie is in your browser.
2. You visit `evil.com` in another tab.
3. `evil.com` has `<form action="https://bank.com/transfer" method="POST">` that auto-submits.
4. Your browser sends the form to `bank.com` **with your session cookie attached** — because cookies travel on every request to their origin, regardless of which page initiated it.

That's the CSRF attack. The fix is to require *something the attacker can't include*:

- A **CSRF token** in a form field or header, set as a cookie or returned by a previous GET; the server compares the cookie value to the header/form value (the "double-submit cookie" pattern).
- The **`SameSite=Lax` cookie attribute**, which tells the browser to omit the cookie on cross-site requests for unsafe methods. Modern browsers default to Lax. With `SameSite=Strict`, the cookie is omitted on *any* cross-site navigation, even GETs.

**The key insight: CSRF only threatens cookie-based auth.** Bearer-token APIs are immune by construction, because:

- The token sits in `localStorage` (or memory), not a cookie.
- `evil.com`'s JavaScript can't read it (same-origin policy on `localStorage`).
- An auto-submitted form from `evil.com` carries no `Authorization` header.

**So which do you use?**

- **Bearer tokens (notebooks 5.1–5.2)**: no CSRF, but tokens are exposed to any XSS in your frontend. You trade one risk for another.
- **Cookies + `SameSite=Lax` + CSRF token**: protected against CSRF and (with `HttpOnly`) against XSS exfiltration of the credential. More wiring; standard for browser apps.

Below: the safe cookie setup, plus a TestClient demonstration of `Lax` vs `Strict`. We use a `"session"` cookie holding an opaque session id (or a JWT — same wiring).

In [ ]:
import secrets
from fastapi import Cookie, HTTPException, Response, status

csrf_app = FastAPI()
SESSIONS = {"valid-session-1": "alice"}  # stand-in for a real session store

@csrf_app.post("/login")
def login(response: Response):
    # Issue a session cookie with the three security attributes that matter.
    response.set_cookie(
        key="session",
        value="valid-session-1",
        httponly=True,   # JavaScript on your own page cannot read it -> XSS can't steal it
        secure=True,     # only sent over HTTPS
        samesite="lax",  # default in modern browsers; blocks CSRF for unsafe methods
        max_age=3600,
    )
    return {"login": "ok"}

# Double-submit CSRF token: GET issues both a cookie and a header value;
# POST requires the client to echo the header value, server compares.
@csrf_app.get("/csrf")
def issue_csrf(response: Response):
    token = secrets.token_urlsafe(32)
    response.set_cookie(key="csrf", value=token, samesite="lax", secure=True, max_age=3600)
    return {"csrf_token": token}

@csrf_app.post("/transfer")
def transfer(
    session: str | None = Cookie(default=None),
    csrf: str | None = Cookie(default=None),
    x_csrf_token: str | None = None,  # would be Header(default=None) in a real route
):
    if not session or SESSIONS.get(session) is None:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Not logged in")
    if not csrf or csrf != x_csrf_token:
        raise HTTPException(status_code=status.HTTP_403_FORBIDDEN, detail="CSRF token mismatch")
    return {"transfer": "ok", "by": SESSIONS[session]}

# What we're showing: the *server* setting these cookie attributes. The actual CSRF protection
# is enforced by the BROWSER honoring SameSite — TestClient won't drop cookies for us. So we
# inspect the response headers and trust the browser to do the rest.
c = TestClient(csrf_app)
r = c.post("/login")
cookie_header = r.headers.get("set-cookie")
print("Set-Cookie on /login:")
print(" ", cookie_header)
for attr in ("HttpOnly", "Secure", "SameSite=lax", "Max-Age=3600"):
    print(f"  contains {attr!r:20s} ->", attr.lower() in cookie_header.lower())

Three cookie attributes you should never ship without:

- **`HttpOnly`** — JavaScript on your *own* page cannot read the cookie. Any XSS in your frontend can't exfiltrate the session.
- **`Secure`** — the cookie is only sent over HTTPS. Stops the cookie from leaking on an http:// downgrade.
- **`SameSite=Lax`** — modern default; blocks cookies on cross-site `POST`/`PUT`/`DELETE`. **This alone kills the auto-submit CSRF.** Use `Strict` for the most sensitive flows (e.g., admin), accept that bookmarked deep-links won't carry the cookie.

Combine with the **double-submit CSRF token** pattern (issue a token in a cookie + require the client to echo it in a header) and you've covered the realistic attack surface for browser-cookie auth.

If you're building a pure-API serving SPAs and mobile, **stick with Bearer tokens** (notebooks 5.1–5.2). The CSRF wiring above is the cost of accepting cookie auth in the browser — pay it deliberately.

## 4. Rate Limiting with `slowapi`

Rate limiting answers "how many requests per minute is *this client* allowed?" without leaning on full auth on every hit. Three reasons to add it to every public-facing API:

- **Brute-force protection.** A `/token` endpoint with no limit is a free password-guessing oracle.
- **Cost control.** Each request costs CPU and downstream calls. Misbehaving clients shouldn't be able to consume all of it.
- **Backpressure on hot endpoints.** A market-data fetch might be cheap, but at 10k rps it isn't.

`slowapi` is FastAPI-flavored `flask-limiter`: a `Limiter` keyed on the client (by default, remote IP), plus a `@limiter.limit("5/minute")` decorator per route. On a breach, it raises `RateLimitExceeded`, mapped to **429 Too Many Requests** with a `Retry-After` header.

In [ ]:
from fastapi import Request
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.errors import RateLimitExceeded
from slowapi.util import get_remote_address

# Key the limiter on the client IP. With auth, swap this for a user-id getter (section 5).
limiter = Limiter(key_func=get_remote_address)

rl_app = FastAPI()
rl_app.state.limiter = limiter
rl_app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

@rl_app.get("/prices/{ticker}")
@limiter.limit("3/minute")  # 3 requests per minute per IP
def get_price(request: Request, ticker: str):
    return {"ticker": ticker, "price": 100.0}

rl_client = TestClient(rl_app)

# Burst three requests — all 200.
for i in range(3):
    r = rl_client.get("/prices/AAPL")
    print(f"req {i+1}: {r.status_code}")

# The fourth one breaks the budget.
r = rl_client.get("/prices/AAPL")
print(f"req 4: {r.status_code} | body: {r.json()} | Retry-After: {r.headers.get('retry-after')}")

Three slowapi quirks worth knowing:

- **The decorated route must accept `request: Request` as a parameter.** `slowapi` reads the request to pull the key (IP or user-id). Forget it and you get a confusing decoration error.
- **`429` is the right status.** Not 503 ("server is dead"), not 403 ("forbidden forever"). 429 says "try again later," with `Retry-After` telling the client *how* long.
- **State is in-process.** `limiter` defaults to an in-memory counter. Restart the worker, counters reset. Run two workers, each enforces its own counter — so the effective limit is `N × your_setting`. Section 6 fixes that with Redis.

## 5. Per-Route and Per-User Limits (and Bypass for `/health`)

Different routes deserve different limits. `/token` (password attempts) wants something strict like `5/minute`; market-data reads can take `60/minute`; `/health` should never be limited because your load balancer hits it constantly. And once you have an authenticated user, you'd rather rate-limit *per user* than per IP — the same NAT'd corporate gateway shouldn't take down the limit for everyone behind it.

Patterns:

- **Per-route limits** — stack different `@limiter.limit("...")` lines on different routes.
- **Per-user limits** — define a custom `key_func` that pulls the user id from the request (after auth runs). With a `Limiter(key_func=user_key)`, the counter is per-user.
- **Bypass `/health`** — pass `@limiter.exempt` on the route, or skip the decorator entirely (no decorator → no limit).

In [ ]:
# Per-user key: read a fake X-User-Id header. In a real app this comes from get_current_user.
def user_key(request: Request) -> str:
    return request.headers.get("X-User-Id") or get_remote_address(request)

user_limiter = Limiter(key_func=user_key)

app2 = FastAPI()
app2.state.limiter = user_limiter
app2.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

@app2.get("/health")
@user_limiter.exempt  # explicit: load balancer can hammer this endpoint freely
def health():
    return {"status": "ok"}

@app2.post("/token")
@user_limiter.limit("2/minute")  # cheap brute-force defense on the password endpoint
def token(request: Request):
    return {"access_token": "..."}

@app2.get("/prices/{ticker}")
@user_limiter.limit("5/minute")  # market data — a bit more generous
def price(request: Request, ticker: str):
    return {"ticker": ticker, "price": 100.0}

c2 = TestClient(app2)

# /health — no decorator, no counter. Loop it 20 times: all 200.
health_codes = {c2.get("/health").status_code for _ in range(20)}
print("health (20 hits)        :", health_codes)

# /token — 2/minute. Three tries from the same user -> third is 429.
for i in range(3):
    r = c2.post("/token", headers={"X-User-Id": "alice"})
    print(f"  alice token {i+1}        : {r.status_code}")

# Same instant, DIFFERENT user -> his counter is independent.
r = c2.post("/token", headers={"X-User-Id": "bob"})
print(f"  bob token 1           : {r.status_code}")

# /prices — separate budget from /token. Verify alice can still query prices even though
# her /token budget is exhausted.
for i in range(2):
    r = c2.get("/prices/AAPL", headers={"X-User-Id": "alice"})
    print(f"  alice price {i+1}        : {r.status_code}")

Three patterns you'll see again in 7.3 and 8.x:

- **One Limiter, many limits.** Stack different `@limiter.limit("...")` strings on different routes. Each route has its own counter.
- **Custom `key_func` is where per-user, per-API-key, or per-tenant limiting lives.** Read whatever identifier you trust from the request; default to IP for unauthenticated hits.
- **`@limiter.exempt` for `/health`** (and `/metrics`, `/openapi.json` — anything ops or browsers poll constantly). Don't make a load balancer health check the thing that takes you down.

## 6. What a Real Rate Limiter Looks Like (Redis-backed)

Everything above runs in **process memory**. Two failure modes that disqualify it for production:

- **Multiple workers / replicas.** `gunicorn -w 4` runs four worker processes; behind a load balancer you might have N replicas. Each has its own counter. A user limited to `5/minute` per worker can get `4N × 5/minute` in the worst case — the *real* limit is `N` times whatever you typed.
- **Restarts wipe state.** Deploy → all counters reset to zero. An attacker that knows your deploy cadence can wait for it.

The fix: store counters in a **shared backend**, typically Redis. `slowapi` supports this with one keyword:

```python
limiter = Limiter(
    key_func=user_key,
    storage_uri="redis://redis.internal:6379",
    strategy="fixed-window-elastic-expiry",  # or "moving-window"
)
```

What you lose if you skip Redis:

| Concern              | In-memory               | Redis-backed             |
|----------------------|-------------------------|--------------------------|
| Per-replica accuracy | Off by `N` (worker count)| Exact across replicas    |
| Persistence on restart| Lost                   | Survives a process bounce|
| Operational overhead | None                    | One Redis to babysit     |
| Latency cost         | Microseconds            | Sub-millisecond network  |

If your rate-limit logic is just "keep cheap clients honest" — in-memory is fine. If it's a **security control** (`/token`, `/login`, anything brute-forceable) — Redis or a managed gateway (API Gateway, Cloudflare) is the right call. Don't ship a security control whose effective threshold is N× what you advertise.

For really serious traffic, the rate limiter moves *out* of the app altogether and into the edge: NGINX `limit_req`, Envoy ratelimit filter, or a CDN/WAF layer. The application-level limit becomes a backstop, not the primary defense. We won't wire that up here — it's a deployment concern (notebook 8.x) — but the takeaway is: **for the capstone, slowapi + Redis is the right shape; for a serious public API, push the limiter outside the app.**

## Key Takeaways

- **CORS is a browser policy.** Set explicit `allow_origins`, set `allow_credentials=True` if you send cookies or `Authorization`, and never use `*` with credentials. TestClient won't catch CORS bugs — test from a browser.
- **Preflight `OPTIONS`** is the request to debug when CORS "doesn't work". Custom headers in `allow_headers`, non-simple methods in `allow_methods`, `max_age` to cache the result.
- **CSRF only matters for cookie auth.** Bearer-token APIs are immune by construction. With cookies: `HttpOnly`, `Secure`, `SameSite=Lax`, plus a double-submit token for unsafe methods.
- **Rate limit everything public, especially `/token`.** `slowapi` + `@limiter.limit("5/minute")` gets you a baseline.
- **Key the limiter on whatever you trust.** Per-IP for anonymous hits, per-user/api-key once auth ran. `@limiter.exempt` for `/health` and other ops endpoints.
- **In-memory limiters lie at scale.** Multiple workers and restarts wipe counters; the real ceiling is `N × your_setting`. Use Redis (`storage_uri=`) for any threshold that's a *security* control, or push the limiter to the edge.
- **Capstone tie-in**: `main.py` will install `CORSMiddleware` with origins from `settings.cors_origins`; `auth.py` will rate-limit `/token` at `5/minute` per IP; `routers/portfolios.py` will exempt `/health`. CSRF stays out of the capstone — it's a Bearer-token API.

## Exercises

**1. Catch a CORS misconfiguration.** Configure `CORSMiddleware` with `allow_origins=["*"]` and `allow_credentials=True`. Send a preflight from `https://app.example.com`. Inspect the response headers — `Allow-Origin` will be `"*"` *but* `Allow-Credentials` will be `"true"`. Note in a markdown cell why a real browser would refuse to send the credentialed real request anyway. Then fix the config to an explicit origin and confirm the headers change.

**2. Simulate a rate-limit breach.** Add a `/token` endpoint with `@limiter.limit("3/minute")` keyed on remote IP. Burst 4 requests from TestClient. Assert that the first 3 return 200 and the 4th returns 429 with a `Retry-After` header. Now add a second test from a different `X-Forwarded-For` (by switching to a `key_func` that reads it) and confirm the second "client" gets its own fresh budget.

**3. Endpoint-class limits.** Split the API into three classes: `/auth/*` (5/min), `/read/*` (60/min), `/health` (exempt). Use `APIRouter`s from chapter 2, attach a different `@limiter.limit` to each router-level dependency. Verify each class enforces its own ceiling independently. Note in a markdown cell why you'd *not* set a global limit covering all routes — what you'd lose.